In [ ]:
import pandas as pd
import numpy as np

# 1. Carrega os metadados (assumindo que 'metadata.csv' ou 'METADADOS_ATUALIZADO' tem Sector, Industry e mcap_YEAR)
temp = pd.read_csv("../../data/02_clean/metadata - metadata_att.csv") 

# 2. Carrega as métricas que já estão no formato longo (node, year, hrm, pozzi, beta, momentum)
df_metrics = pd.read_parquet("../../data/02_clean/df_metrics.parquet")
df_metrics["hrm"] = -df_metrics["hrm"]

# 3. Cria os decis dinamicamente ano a ano para evitar vazamento de dados
decile_labels = [f"decil_{i}" for i in range(1, 11)]

def create_deciles(series):
    return pd.qcut(series, q=10, labels=decile_labels, duplicates='drop')

df_metrics['decil_hrm'] = df_metrics.groupby('year')['hrm'].transform(create_deciles)
df_metrics['decil_pozzi'] = df_metrics.groupby('year')['pozzi'].transform(create_deciles)

years = range(2014, 2025)
metrics_to_run = ["hrm", "pozzi"]

for metric in metrics_to_run:
    all_years_df = []
    
    for year in years:
        print(f"Processing year {year}, metric={metric}")
        
        # Filtra os dados apenas para o ano em loop
        df_year = df_metrics[df_metrics['year'] == f'{year}'].copy()
        
        # Renomeia 'node' para 'Ticker' para conseguir dar merge
        df_year = df_year.rename(columns={'node': 'Ticker'})
        
        # Define o nome do portfolio como sendo o decil que o ativo caiu
        df_year['portfolio'] = df_year[f'decil_{metric}']
        
        # Cruza com temp para buscar Sector, Industry e mcap_20XX
        final_df = df_year.merge(temp, on="Ticker", how="inner")
        
        # Cria as colunas com sufixo do ano (para reproduzir seu DataFrame esparso)
        final_df[f'beta_{year}'] = final_df['beta']
        final_df[f'momentum_{year}'] = final_df['momentum']
        
        # Filtra e limpa as colunas. Removemos a parte do 'cut' daqui.
        cols_to_keep = [
            "Ticker", "Sector", "Industry", "portfolio", "year", 
            f"mcap_{year}", f"beta_{year}", f"momentum_{year}"
        ]
        
        # Se 'temp' já tinha problemas de vírgulas no mcap, você pode incluir o replace aqui, 
        # caso contrário o merge acima já traz certinho.
        if f'mcap_{year}' in final_df.columns:
            # (Opcional) Limpa vírgula se ainda for string
            if final_df[f'mcap_{year}'].dtype == object:
                final_df[f'mcap_{year}'] = final_df[f'mcap_{year}'].astype(str).str.replace(",", ".", regex=False).apply(pd.to_numeric, errors="coerce")
        
        # Mantém só as colunas que de fato existem após o merge
        cols_to_keep = [c for c in cols_to_keep if c in final_df.columns]
        
        final_df = final_df[cols_to_keep]
        all_years_df.append(final_df)
        
    # Concatena os anos. Linhas de 2015 terão NaN nas colunas de beta_2023, etc.
    final_panel_df = pd.concat(all_years_df, ignore_index=True)
    
    # Salva gerando complete_metadata_hrm.csv e complete_metadata_pozzi.csv
    output_path = f"../../data/07_portfolios_metadata/complete_metadata_{metric}.parquet"
    final_panel_df.to_parquet(output_path, index=False)
    print(f"Salvo: complete_metadata_{metric}.csv\n")

Processing year 2014, metric=hrm
Processing year 2015, metric=hrm
Processing year 2016, metric=hrm
Processing year 2017, metric=hrm
Processing year 2018, metric=hrm
Processing year 2019, metric=hrm
Processing year 2020, metric=hrm
Processing year 2021, metric=hrm
Processing year 2022, metric=hrm
Processing year 2023, metric=hrm
Processing year 2024, metric=hrm
Salvo: complete_metadata_hrm.csv

Processing year 2014, metric=pozzi
Processing year 2015, metric=pozzi
Processing year 2016, metric=pozzi
Processing year 2017, metric=pozzi
Processing year 2018, metric=pozzi
Processing year 2019, metric=pozzi
Processing year 2020, metric=pozzi
Processing year 2021, metric=pozzi
Processing year 2022, metric=pozzi
Processing year 2023, metric=pozzi
Processing year 2024, metric=pozzi
Salvo: complete_metadata_pozzi.csv



In [29]:
all_years_df[0]

,Ticker,Sector,Industry,portfolio,year,beta_2014,momentum_2014
0,A,Healthcare,Diagnostics & Research,decil_1,2014,1.289674,0.036170
1,AA,Basic Materials,Aluminum,decil_9,2014,1.470522,0.518100
2,AAL,Industrials,Airlines,decil_8,2014,1.605365,1.117107
3,AAME,Financial,Insurance - Life,decil_8,2014,0.045365,-0.002096
4,AAON,Industrials,Building Products & Equipment,decil_6,2014,1.682752,0.066075
...,...,...,...,...,...,...,...
2718,ZNB,Communication Services,Entertainment,decil_8,2014,0.628092,-0.102083
2719,ZROZ,Financial,Exchange Traded Fund,decil_8,2014,-0.692103,0.467750
2720,ZSL,Financial,Exchange Traded Fund,decil_10,2014,0.112141,0.300398
2721,ZTR,Financial,Closed-End Fund - Equity,decil_1,2014,0.451517,0.094877


In [27]:
df_metrics[df_metrics['year'] == '2014']

,node,hrm,pozzi,degree,closeness,eig,year,beta,momentum,decil_hrm,decil_pozzi
0,A,0.113502,0.162029,2.219453,0.289924,0.000978,2014,1.289674,0.036170,decil_6,decil_1
1,AA,0.242146,0.222550,1.851429,0.271225,0.000676,2014,1.470522,0.518100,decil_7,decil_9
2,AAL,0.411204,0.213583,2.391266,0.237571,0.000902,2014,1.605365,1.117107,decil_8,decil_8
3,AAME,0.566156,0.212748,0.223170,0.226859,0.000381,2014,0.045365,-0.002096,decil_9,decil_8
4,AAON,-0.892336,0.186629,2.481132,0.331172,0.034447,2014,1.682752,0.066075,decil_1,decil_6
...,...,...,...,...,...,...,...,...,...,...,...
2718,ZNB,0.473366,0.216124,0.299424,0.243640,0.000071,2014,0.628092,-0.102083,decil_9,decil_8
2719,ZROZ,0.382266,0.214811,3.360004,0.238718,0.000283,2014,-0.692103,0.467750,decil_8,decil_8
2720,ZSL,0.429236,0.276925,6.440218,0.211660,0.000012,2014,0.112141,0.300398,decil_8,decil_10
2721,ZTR,0.098267,0.105097,0.509073,0.301451,0.001542,2014,0.451517,0.094877,decil_5,decil_1


In [30]:
final_panel_df

,Ticker,Sector,Industry,portfolio,year,beta_2014,momentum_2014,mcap_2015,beta_2015,momentum_2015,...,momentum_2021,mcap_2022,beta_2022,momentum_2022,mcap_2023,beta_2023,momentum_2023,mcap_2024,beta_2024,momentum_2024
0,A,Healthcare,Diagnostics & Research,decil_1,2014,1.289674,0.036170,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AA,Basic Materials,Aluminum,decil_9,2014,1.470522,0.518100,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AAL,Industrials,Airlines,decil_8,2014,1.605365,1.117107,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AAME,Financial,Insurance - Life,decil_8,2014,0.045365,-0.002096,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AAON,Industrials,Building Products & Equipment,decil_6,2014,1.682752,0.066075,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40657,ZTS,Healthcare,Drug Manufacturers - Specialty & Generic,decil_1,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.292747e+10,0.623014,-0.166471
40658,ZUMZ,Consumer Cyclical,Apparel Retail,decil_6,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.539357e+08,1.244622,-0.073231
40659,ZVRA,Healthcare,Biotechnology,decil_3,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.857353e+08,NaN,NaN
40660,ZWS,Industrials,Pollution & Treatment Controls,decil_5,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.353906e+09,NaN,NaN


In [20]:
temp = pd.read_parquet("../../data/02_clean/df_metrics.parquet")
temp[temp["year"]=='2014']

,node,hrm,pozzi,degree,closeness,eig,year,beta,momentum
0,A,-0.113502,0.162029,2.219453,0.289924,0.000978,2014,1.289674,0.036170
1,AA,-0.242146,0.222550,1.851429,0.271225,0.000676,2014,1.470522,0.518100
2,AAL,-0.411204,0.213583,2.391266,0.237571,0.000902,2014,1.605365,1.117107
3,AAME,-0.566156,0.212748,0.223170,0.226859,0.000381,2014,0.045365,-0.002096
4,AAON,0.892336,0.186629,2.481132,0.331172,0.034447,2014,1.682752,0.066075
...,...,...,...,...,...,...,...,...,...
2718,ZNB,-0.473366,0.216124,0.299424,0.243640,0.000071,2014,0.628092,-0.102083
2719,ZROZ,-0.382266,0.214811,3.360004,0.238718,0.000283,2014,-0.692103,0.467750
2720,ZSL,-0.429236,0.276925,6.440218,0.211660,0.000012,2014,0.112141,0.300398
2721,ZTR,-0.098267,0.105097,0.509073,0.301451,0.001542,2014,0.451517,0.094877


In [12]:
final_panel_df.to_csv("../../data/07_portfolios_metadata/complete_metadata.csv")

In [7]:
pd.read_csv(
                    f"../../data/07_portfolios_metadata/central_metadata_{year}_{k}.csv"
                )

,Unnamed: 0,Ticker,Sector,Industry,Country,mcap_2015,mcap_2016,mcap_2017,mcap_2018,mcap_2019,mcap_2020,mcap_2021,mcap_2022,mcap_2023,mcap_2024
0,0,ADI,Technology,Semiconductors,USA,1.721370e+10,22424039320,32860349790,31645778490,4.376176e+10,5.454236e+10,92330399070,8.318306e+10,9.843910e+10,1.054048e+11
1,1,ADX,Financial,Closed-End Fund - Equity,USA,1.256244e+09,1263842922,1528552082,1339604519,3.431935e+09,1.884394e+09,2287898974,1.757893e+09,2.147268e+09,2.244572e+09
2,2,AFG,Financial,Insurance - Property & Casualty,USA,6.268530e+09,7684064000,9581418428,8093382000,9.847548e+09,7.526558e+09,11672200000,1.169681e+10,9.943461e+09,1.148843e+10
3,3,ARW,Technology,Electronics & Computer Distribution,USA,4.958229e+09,6364380600,7085729200,5888330000,6.813689e+09,7.286019e+09,9108876800,6.397070e+09,6.649911e+09,5.996491e+09
4,4,ASB,Financial,Banks - Regional,USA,2.786269e+09,3686549100,3824300200,3244135120,3.409610e+09,2.597653e+09,3360962790,3.450893e+09,3.210318e+09,3.631199e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,75,UNP,Industrials,Railroads,USA,6.600080e+10,84934656000,105080760000,99083264000,1.237146e+11,1.390077e+11,159270146000,1.268718e+11,1.495580e+11,1.385571e+11
76,76,VNO,Real Estate,REIT - Office,USA,1.521511e+10,15959863499,14846460180,11828438670,1.270403e+10,7.147548e+09,8025566640,3.992003e+09,5.377698e+09,8.045237e+09
77,77,WFC,Financial,Banks - Diversified,USA,2.776981e+11,276960816000,299709375310,209733120000,2.208382e+11,1.249844e+11,183816178000,1.568979e+11,1.782207e+11,2.304153e+11
78,78,WOR,Industrials,Metal Fabrication,USA,1.147259e+09,1834810000,1640002280,1213147440,1.428180e+09,1.650516e+09,1676541300,1.488706e+09,2.838078e+09,1.980511e+09


In [8]:
pd.read_csv(
                    f"../../data/07_portfolios_metadata/peripheral_metadata_{year}_{k}.csv"
                )

,Unnamed: 0,Ticker,Sector,Industry,Country,mcap_2015,mcap_2016,mcap_2017,mcap_2018,mcap_2019,mcap_2020,mcap_2021,mcap_2022,mcap_2023,mcap_2024
0,0,AAPL,Technology,Consumer Electronics,USA,5.805540e+11,613796890240,865303303480,737381440960,1.280300e+12,2.223019e+12,2890626871140,2.064941e+12,2.986095e+12,3.754818e+12
1,1,ABM,Industrials,Specialty Business Services,USA,1.611402e+09,2287040000,2485748000,2132104000,2.522799e+09,2.542848e+09,2773715000,2.945046e+09,2.846705e+09,3.208986e+09
2,2,AMD,Technology,Semiconductors,USA,2.275910e+09,10557540000,9920200000,19272240000,5.365620e+10,1.112442e+11,200452700000,1.044740e+11,2.382146e+11,1.956798e+11
3,3,AMGN,Healthcare,Drug Manufacturers - General,USA,1.222345e+11,108487820000,125834860634,121084739999,1.422313e+11,1.326638e+11,123283560000,1.405124e+11,1.540907e+11,1.402243e+11
4,4,ARL,Real Estate,Real Estate Services,USA,8.671871e+07,80209241,199359526,193084707,2.740299e+08,1.760573e+08,204323331,4.235339e+08,2.812071e+08,2.371120e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,75,WLY,Communication Services,Publishing,USA,2.599389e+09,3115608367,3774762664,2680757043,2.719061e+09,2.556229e+09,3189996270,2.219417e+09,1.739733e+09,2.358242e+09
76,76,WMK,Consumer Defensive,Grocery Stores,USA,1.191601e+09,1797891930,1113326555,1285207606,1.089118e+09,1.286015e+09,1772069424,2.213473e+09,1.720424e+09,1.821563e+09
77,77,WMT,Consumer Defensive,Discount Stores,USA,1.952699e+11,211852800000,292230840000,269762400000,3.370019e+11,4.073679e+11,400646610000,3.822389e+11,4.240785e+11,7.254202e+11
78,78,WSM,Consumer Cyclical,Specialty Retail,USA,5.243779e+09,4243179600,4344712900,4017120600,5.681612e+09,7.791473e+09,12518051399,7.624827e+09,1.294277e+10,2.281436e+10
